# Этап 2: Деанонимизация и графовая кластеризация адресов
 
## Описание файла
В данном ноутбуке реализуется математическая модель деанонимизации участников сети Bitcoin на основе теории графов. Из-за отсутствия явных идентификаторов пользователей в блокчейне применяются детерминированные эвристики для объединения разрозненных адресов в единые сущности (Entities).

**Реализованные алгоритмы:**
1. **Эвристика совместного использования входов (Common Input Ownership):** Поиск компонент связности в неориентированном графе транзакций (с использованием библиотеки `networkx`).
2. **Эвристика адреса сдачи (Change Address):** Выявление одноразовых адресов возврата средств, сгенерированных HD-кошельками отправителей.

**Цель:** Сократить размерность графа и трансформировать таблицу адресов в таблицу реальных акторов сети.

**Входные данные:** `../data/blockchain_dataset.csv`  
**Выходные данные:** `../data/blockchain_final_entities.csv`

In [1]:
import pandas as pd

def apply_change_heuristic(csv_path):
    print("1. Загрузка данных с кластерами...")
    df = pd.read_csv(csv_path)
    
    first_seen = df.groupby('Address')['BlockHeight'].min().to_dict()
    tx_groups = df.groupby('TxID')
    
    change_addresses_found = 0
    
    print("2. Поиск адресов сдачи (Change Addresses)...")
    for txid, tx_data in tx_groups:
        inputs = tx_data[tx_data['Type'] == 'INPUT']
        outputs = tx_data[tx_data['Type'] == 'OUTPUT']
        
        if len(outputs) == 2 and len(inputs) > 0:
            sender_entity = inputs.iloc[0]['Entity_ID']
            
            output_addrs = outputs['Address'].tolist()
            out_1, out_2 = output_addrs[0], output_addrs[1]
            
            if out_1 == "Non-standard" or out_2 == "Non-standard":
                continue
                
            current_block = tx_data['BlockHeight'].iloc[0]
            
            out_1_is_new = first_seen.get(out_1) == current_block
            out_2_is_new = first_seen.get(out_2) == current_block
            
            change_address = None
            
            if out_1_is_new and not out_2_is_new:
                change_address = out_1
            elif out_2_is_new and not out_1_is_new:
                change_address = out_2
                
            if change_address:
                old_entity = df.loc[df['Address'] == change_address, 'Entity_ID'].iloc[0]
                df.loc[df['Entity_ID'] == old_entity, 'Entity_ID'] = sender_entity
                change_addresses_found += 1

    print(f"\nНайдено и присоединено адресов сдачи: {change_addresses_found}")
    
    output_file = "blockchain_final_entities.csv"
    df.to_csv(output_file, index=False)
    print(f"Готово! Финальные данные сохранены в {output_file}")
    
    return df

In [2]:
input_file = "blockchain_clustered.csv" 

df_final = apply_change_heuristic(input_file)

print("\nТоп-5 кластеров ПОСЛЕ применения эвристики адреса сдачи:")
cluster_sizes = df_final.groupby('Entity_ID')['Address'].nunique().sort_values(ascending=False)
print(cluster_sizes.head())
print(f"\nВсего уникальных сущностей (Entity) осталось: {df_final['Entity_ID'].nunique()}")

1. Загрузка данных с кластерами...
2. Поиск адресов сдачи (Change Addresses)...

Найдено и присоединено адресов сдачи: 82
Готово! Финальные данные сохранены в blockchain_final_entities.csv

Топ-5 кластеров ПОСЛЕ применения эвристики адреса сдачи:
Entity_ID
Cluster_21    323
Cluster_99     73
Cluster_77     33
Cluster_15     22
Cluster_73     17
Name: Address, dtype: int64

Всего уникальных сущностей (Entity) осталось: 754
